# 01 - Input Layer

**Architecture block 1.** Leaf image + climate data + farm location & metadata.

PlantVillage has no coordinates and no dates. This notebook shows how the
geography and meteorology are attached, and verifies that the result is a
genuine learning problem rather than a giveaway.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

## The farm registry

37 real Indian production districts. These are the nodes of the spatial graph.

In [ ]:
from cropforecast.data.farms import SITES, to_frame, all_crops, sites_for_crop
sites = to_frame()
print(f"{len(sites)} sites across {sites.state.nunique()} states")
print(f"latitude {sites.lat.min():.2f} to {sites.lat.max():.2f}")
sites.head(10)

In [ ]:
for c in all_crops():
    print(f"{c:28s} {len(sites_for_crop(c)):2d} sites")

## The PlantVillage index

Note `leaf_group`: PlantVillage photographs the same physical leaf several times.

In [ ]:
from cropforecast.data.plantvillage import build_index, class_table
index = build_index(cfg.paths.raw_images, cfg.paths.splits, cfg.paths.leaf_map)
print(f"{len(index):,} images | {index.class_name.nunique()} classes | "
      f"{index.leaf_group.nunique():,} leaf groups")
print(f"average {len(index)/index.leaf_group.nunique():.2f} photographs per physical leaf")
class_table(index).head(12)

### Why leaf grouping matters

A random split scatters near-duplicate photographs of the *same leaf* across
train and test. Measure how bad it is.

In [ ]:
from cropforecast.data.splits import leaf_disjoint_split, random_split, leakage_report
import pandas as pd
obs = pd.read_parquet(Path(cfg.paths.processed) / "observations.parquet")
print("random split      :", leakage_report(random_split(obs, seed=cfg.project.seed)))
print("leaf-disjoint     :", leakage_report(leaf_disjoint_split(obs, seed=cfg.project.seed)))

## Real climate

ERA5 reanalysis pulled once from the Open-Meteo archive and cached.

In [ ]:
from cropforecast.data.climate import fetch_all, coverage_report
climate_raw = fetch_all("2020-12-01", "2023-12-31", cfg.paths.climate,
                        daily_vars=list(cfg.climate.daily_vars), verbose=False)
print(f"{len(climate_raw):,} site-days")
coverage_report(climate_raw).head(12)

## Does the assignment create real signal?

Mean weather per disease class. Note the opposite responses.

In [ ]:
from cropforecast.data.assign import assignment_report
rep = assignment_report(obs)
pd.concat([rep.head(6), rep.tail(6)])[["n","temp_C","rh_pct","rain_mm","lwd_h","vpd"]]

**Read the table.** Late blight and strawberry leaf scorch sit at ~16 C with
10-12 h of leaf wetness. Spider mites and corn sit at ~28 C with high VPD. That
contrast is what the climate branch learns from.